In [1]:
import os
import random
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow : {tf.__version__}")
print(f"GPU        : {tf.config.list_physical_devices('GPU')}")
print(f"Seed       : {SEED} ✅")

TensorFlow : 2.20.0
GPU        : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Seed       : 42 ✅


In [11]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = "/content/drive/MyDrive/cats_vs_dogs_classifier"
PROCESSED_DIR = Path(f"{DRIVE_PATH}/data/processed")
MODEL_DIR = Path(f"{DRIVE_PATH}/models")
DOCS_DIR = Path(f"{DRIVE_PATH}/docs")

MODEL_DIR.mkdir(parents=True, exist_ok=True)
DOCS_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE
CLASS_NAMES = ["cat","dog"]
INPUT_SHAPE = (224,224, 3)

print(f"Processed data : {PROCESSED_DIR}")
print(f"Models saved to : {MODEL_DIR}")
print(f"Docs saved to : {DOCS_DIR}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Processed data : /content/drive/MyDrive/cats_vs_dogs_classifier/data/processed
Models saved to : /content/drive/MyDrive/cats_vs_dogs_classifier/models
Docs saved to : /content/drive/MyDrive/cats_vs_dogs_classifier/docs


# Rebuild Datasets

In [12]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.20),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.10, 0.10),
], name="data_augmentation")

def preprocess_image(image, label):
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def build_dataset(directory, training=False):
    ds = tf.keras.utils.image_dataset_from_directory(
        directory,
        labels="inferred", label_mode="binary",
        class_names=CLASS_NAMES,
        image_size=IMG_SIZE, batch_size=BATCH_SIZE,
        shuffle=training, seed=SEED,
    )
    ds = ds.map(preprocess_image, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(
            lambda x, y: (data_augmentation(x, training=True), y),
            num_parallel_calls=AUTOTUNE,
        )
    return ds.cache().prefetch(AUTOTUNE)

train_ds = build_dataset(str(PROCESSED_DIR / "train"), training=True)
val_ds   = build_dataset(str(PROCESSED_DIR / "val"),   training=False)
test_ds  = build_dataset(str(PROCESSED_DIR / "test"),  training=False)

# Quick check
for images, labels in train_ds.take(1):
    print(f"train_ds batch : {images.shape}  labels: {labels.numpy().flatten()[:6].tolist()}")
for images, labels in val_ds.take(1):
    print(f"val_ds   batch : {images.shape}")
print("Datasets ready ✅")

Found 16000 files belonging to 2 classes.
Found 4000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
train_ds batch : (32, 224, 224, 3)  labels: [1.0, 0.0, 1.0, 1.0, 0.0, 0.0]
val_ds   batch : (32, 224, 224, 3)
Datasets ready ✅


# Rebuild Models

In [13]:
# ── Custom CNN ────────────────────────────────────────────────────────────────
def build_custom_cnn():
    inputs = keras.Input(shape=INPUT_SHAPE, name="input")
    x = layers.Conv2D(32, (3,3), padding="same")(inputs)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv2D(32, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x); x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(64, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv2D(64, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x); x = layers.Dropout(0.25)(x)

    x = layers.Conv2D(128, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.Conv2D(128, (3,3), padding="same")(x)
    x = layers.BatchNormalization()(x); x = layers.Activation("relu")(x)
    x = layers.MaxPooling2D()(x); x = layers.Dropout(0.25)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256)(x); x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x); x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="custom_cnn")


# ── Transfer learning wrapper ─────────────────────────────────────────────────
def build_transfer_model(base_model, name, dropout=0.5):
    base_model.trainable = False
    inputs  = keras.Input(shape=INPUT_SHAPE)
    x       = base_model(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dropout(dropout)(x)
    x       = layers.Dense(256, activation="relu")(x)
    x       = layers.Dropout(dropout * 0.5)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name=name)


# ── Phase 2 unfreeze helper ───────────────────────────────────────────────────
def unfreeze_top_layers(model, n=20):
    base = model.layers[1]
    base.trainable = True
    for layer in base.layers[:-n]:
        layer.trainable = False
    trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
    print(f"  Unfrozen top {n} layers — trainable params: {trainable:,}")


# ── Build all 4 ───────────────────────────────────────────────────────────────
cnn          = build_custom_cnn()
vgg16        = build_transfer_model(keras.applications.VGG16(include_top=False, weights="imagenet", input_shape=INPUT_SHAPE),        "vgg16")
resnet50     = build_transfer_model(keras.applications.ResNet50(include_top=False, weights="imagenet", input_shape=INPUT_SHAPE),      "resnet50")
efficientnet = build_transfer_model(keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=INPUT_SHAPE),"efficientnetb0", dropout=0.4)

print("All 4 models built ✅")
print(f"  Custom CNN      : {cnn.count_params():>12,} params")
print(f"  VGG16           : {vgg16.count_params():>12,} params")
print(f"  ResNet50        : {resnet50.count_params():>12,} params")
print(f"  EfficientNetB0  : {efficientnet.count_params():>12,} params")

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
All 4 models built ✅
  Custom CNN      :      323,105 params
  VGG16           :   14,848,321 params
  ResNet50        :   24,120,705 params
  EfficientNetB0  :    4,382,884 params


# Training Utils

In [14]:
def get_callbacks(model_name):
    """Returns 3 callbacks for any model."""
    checkpoint_path = str(MODEL_DIR / f"{model_name}_best.h5")

    return [
        # Save best weights automatically
        keras.callbacks.ModelCheckpoint(
            filepath       = checkpoint_path,
            monitor        = "val_accuracy",
            save_best_only = True,
            verbose        = 1,
        ),
        # Stop if no improvement for 5 epochs — go back to best weights
        keras.callbacks.EarlyStopping(
            monitor              = "val_accuracy",
            patience             = 5,
            restore_best_weights = True,
            verbose              = 1,
        ),
        # Halve LR if val_loss stuck for 3 epochs
        keras.callbacks.ReduceLROnPlateau(
            monitor  = "val_loss",
            factor   = 0.5,
            patience = 3,
            min_lr   = 1e-7,
            verbose  = 1,
        ),
    ]


def compile_model(model, lr):
    """Compile with Adam + binary crossentropy + all metrics."""
    model.compile(
        optimizer = keras.optimizers.Adam(learning_rate=lr),
        loss      = "binary_crossentropy",
        metrics   = [
            "accuracy",
            keras.metrics.AUC(name="auc"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
        ],
    )
    print(f"  Compiled — optimizer: Adam(lr={lr})  loss: binary_crossentropy")


print("Callbacks and compile ready ✅")

Callbacks and compile ready ✅


In [15]:
def plot_history(history_dict, model_name, save=True):
    """Plot accuracy + loss curves. Optionally save to Drive."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history_dict["accuracy"]) + 1)

    # Accuracy
    axes[0].plot(epochs, history_dict["accuracy"],     "b-o", markersize=4, label="Train")
    axes[0].plot(epochs, history_dict["val_accuracy"], "r-o", markersize=4, label="Val")
    axes[0].fill_between(epochs,
                         history_dict["accuracy"],
                         history_dict["val_accuracy"],
                         alpha=0.08, color="purple", label="Gap")
    axes[0].set_title(f"{model_name} — Accuracy", fontsize=12, fontweight="bold")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Accuracy")
    axes[0].legend(); axes[0].grid(True, alpha=0.3); axes[0].set_ylim([0, 1])

    # Loss
    axes[1].plot(epochs, history_dict["loss"],     "b-o", markersize=4, label="Train")
    axes[1].plot(epochs, history_dict["val_loss"], "r-o", markersize=4, label="Val")
    axes[1].set_title(f"{model_name} — Loss", fontsize=12, fontweight="bold")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Loss")
    axes[1].legend(); axes[1].grid(True, alpha=0.3)

    plt.suptitle(f"Training Curves: {model_name}", fontsize=14, y=1.01)
    plt.tight_layout()

    if save:
        path = DOCS_DIR / f"{model_name.lower().replace(' ','_')}_curves.png"
        plt.savefig(path, dpi=120, bbox_inches="tight")
        print(f"  Saved → {path.name}")

    plt.show()

# Store all histories for comparison at the end
all_histories = {}

print("Plot helper ready ✅")

Plot helper ready ✅


# Train Model 1 - Custom CNN

In [16]:
print("=" * 50)
print("  Training: Custom CNN")
print("=" * 50)

compile_model(cnn, lr=1e-3)

history_cnn = cnn.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = 30,
    callbacks       = get_callbacks("cnn"),
    verbose         = 1,
)

all_histories["Custom CNN"] = history_cnn.history

print(f"\nBest val_accuracy: {max(history_cnn.history['val_accuracy']):.4f}")
print(f"Epochs trained   : {len(history_cnn.history['accuracy'])}")
print(f"Weights saved to : {MODEL_DIR}/cnn_best.h5")

  Training: Custom CNN
  Compiled — optimizer: Adam(lr=0.001)  loss: binary_crossentropy
Epoch 1/30
 39/500 ━━━━━━━━━━━━━━━━━━━━ 31:11 4s/step - accuracy: 0.5334 - auc: 0.5475 - loss: 0.8129 - precision: 0.5441 - recall: 0.4526

InvalidArgumentError: Graph execution error:

Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
Detected at node decode_image/DecodeImage defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) INVALID_ARGUMENT:  Input size should match (header_size + row_size * abs_height) but they differ by 2
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]]
	 [[IteratorGetNext/_2]]
  (1) INVALID_ARGUMENT:  Input size should match (header_size + row_size * abs_height) but they differ by 2
	 [[{{node decode_image/DecodeImage}}]]
	 [[IteratorGetNext]]
0 successful operations.
0 derived errors ignored. [Op:__inference_multi_step_on_iterator_14105]

# Train Model 2 - VGG16

In [ ]:
print("=" * 50)
print("  VGG16 — Phase 1: Training head (frozen base)")
print("=" * 50)

compile_model(vgg16, lr=1e-3)

h1_vgg16 = vgg16.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = 5,
    callbacks       = get_callbacks("vgg16_phase1"),
    verbose         = 1,
)

print(f"\nPhase 1 best val_accuracy: {max(h1_vgg16.history['val_accuracy']):.4f}")

In [ ]:
print("=" * 50)
print("  VGG16 — Phase 2: Fine-tuning top 20 layers")
print("=" * 50)
print()

unfreeze_top_layers(vgg16, n=20)
compile_model(vgg16, lr=1e-5)   # MUST recompile after unfreezing

h2_vgg16 = vgg16.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = 15,
    callbacks       = get_callbacks("vgg16_phase2"),
    verbose         = 1,
)

# Merge both phases into one history for plotting
merged_vgg16 = {
    k: h1_vgg16.history[k] + h2_vgg16.history.get(k, [])
    for k in h1_vgg16.history
}
all_histories["VGG16"] = merged_vgg16

print(f"\nPhase 2 best val_accuracy: {max(h2_vgg16.history['val_accuracy']):.4f}")
print(f"Overall best val_accuracy: {max(merged_vgg16['val_accuracy']):.4f}")
print(f"Total epochs             : {len(merged_vgg16['accuracy'])}")

In [ ]:
plot_history(all_histories["VGG16"], "VGG16")

# Mark the Phase 1 / Phase 2 boundary on the plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(merged_vgg16["accuracy"]) + 1)
phase1_end = len(h1_vgg16.history["accuracy"])

for ax, metric, val_metric in [
    (axes[0], "accuracy", "val_accuracy"),
    (axes[1], "loss",     "val_loss"),
]:
    ax.plot(epochs, merged_vgg16[metric],     "b-o", markersize=3, label="Train")
    ax.plot(epochs, merged_vgg16[val_metric], "r-o", markersize=3, label="Val")
    ax.axvline(x=phase1_end + 0.5, color="green", linestyle="--", linewidth=1.5,
               label=f"Phase 1→2 (epoch {phase1_end})")
    ax.set_title(f"VGG16 — {metric.capitalize()}", fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.legend(); ax.grid(True, alpha=0.3)
    if metric == "accuracy": ax.set_ylim([0, 1])

plt.suptitle("VGG16 — Phase 1 vs Phase 2 boundary", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(DOCS_DIR / "vgg16_phases.png", dpi=120, bbox_inches="tight")
plt.show()
print("Phase boundary plot saved ✅")

#Train model 3 - ResNet 50

In [ ]:
print("=" * 50)
print("  ResNet50 — Phase 1: Training head (frozen base)")
print("=" * 50)

compile_model(resnet50, lr=1e-3)

h1_resnet = resnet50.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = 5,
    callbacks       = get_callbacks("resnet50_phase1"),
    verbose         = 1,
)

print(f"\nPhase 1 best val_accuracy: {max(h1_resnet.history['val_accuracy']):.4f}")

In [ ]:
print("=" * 50)
print("  ResNet50 — Phase 2: Fine-tuning top 20 layers")
print("=" * 50)

unfreeze_top_layers(resnet50, n=20)
compile_model(resnet50, lr=1e-5)

h2_resnet = resnet50.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = 15,
    callbacks       = get_callbacks("resnet50_phase2"),
    verbose         = 1,
)

merged_resnet = {
    k: h1_resnet.history[k] + h2_resnet.history.get(k, [])
    for k in h1_resnet.history
}
all_histories["ResNet50"] = merged_resnet

print(f"\nBest val_accuracy: {max(merged_resnet['val_accuracy']):.4f}")
print(f"Total epochs     : {len(merged_resnet['accuracy'])}")

In [ ]:
plot_history(all_histories["ResNet50"], "ResNet50")

# Train Model 4 - EfficientNet B0

In [ ]:
print("=" * 50)
print("  EfficientNetB0 — Phase 1: Training head (frozen base)")
print("=" * 50)

compile_model(efficientnet, lr=1e-3)

h1_effnet = efficientnet.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = 5,
    callbacks       = get_callbacks("efficientnet_phase1"),
    verbose         = 1,
)

print(f"\nPhase 1 best val_accuracy: {max(h1_effnet.history['val_accuracy']):.4f}")

In [ ]:
print("=" * 50)
print("  EfficientNetB0 — Phase 2: Fine-tuning top 20 layers")
print("=" * 50)

unfreeze_top_layers(efficientnet, n=20)
compile_model(efficientnet, lr=1e-5)

h2_effnet = efficientnet.fit(
    train_ds,
    validation_data = val_ds,
    epochs          = 15,
    callbacks       = get_callbacks("efficientnet_phase2"),
    verbose         = 1,
)

merged_effnet = {
    k: h1_effnet.history[k] + h2_effnet.history.get(k, [])
    for k in h1_effnet.history
}
all_histories["EfficientNetB0"] = merged_effnet

print(f"\nBest val_accuracy: {max(merged_effnet['val_accuracy']):.4f}")
print(f"Total epochs     : {len(merged_effnet['accuracy'])}")

In [ ]:
plot_history(all_histories["EfficientNetB0"], "EfficientNetB0")

# All models -- Training Curves side by side

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Training Curves — All 4 Models", fontsize=15, fontweight="bold")

colours = {"Custom CNN": "steelblue", "VGG16": "tomato",
           "ResNet50": "seagreen", "EfficientNetB0": "darkorange"}

for ax, (name, history) in zip(axes.flat, all_histories.items()):
    epochs = range(1, len(history["accuracy"]) + 1)
    colour = colours[name]
    ax.plot(epochs, history["accuracy"],     "-o", color=colour,
            markersize=3, label="Train acc")
    ax.plot(epochs, history["val_accuracy"], "--o", color=colour,
            markersize=3, alpha=0.6, label="Val acc")
    best = max(history["val_accuracy"])
    ax.axhline(best, color="gray", linestyle=":", linewidth=1,
               label=f"Best val: {best:.4f}")
    ax.set_title(name, fontsize=12, fontweight="bold")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3); ax.set_ylim([0, 1])

plt.tight_layout()
plt.savefig(DOCS_DIR / "all_models_curves.png", dpi=120, bbox_inches="tight")
plt.show()
print("Comparison plot saved ✅")

# Training Summary

In [ ]:
print("=" * 65)
print("  TRAINING SUMMARY")
print("=" * 65)
print(f"  {'Model':<18} {'Epochs':>7} {'Best Val Acc':>13} {'Best Val AUC':>13}")
print(f"  {'-'*18} {'-'*7} {'-'*13} {'-'*13}")

summary = {}
for name, history in all_histories.items():
    best_acc = max(history["val_accuracy"])
    best_auc = max(history["val_auc"]) if "val_auc" in history else 0.0
    epochs   = len(history["accuracy"])
    star = " ⭐" if name == "VGG16" else ""
    print(f"  {name+star:<20} {epochs:>7} {best_acc:>13.4f} {best_auc:>13.4f}")
    summary[name] = {"epochs": epochs, "best_val_accuracy": round(best_acc,4),
                     "best_val_auc": round(best_auc, 4)}

print("=" * 65)
print("  ⭐ = production model")

# Save summary to Drive
summary_path = DOCS_DIR / "training_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"\nSummary saved → {summary_path}")

# Verify Saved Weights

In [ ]:
print("Checking saved model files on Drive:\n")

saved_models = list(MODEL_DIR.glob("*_best.h5"))

if not saved_models:
    print(" No model files found — check training completed without errors.")
else:
    for path in sorted(saved_models):
        size_mb = path.stat().st_size / 1024 / 1024
        print(f"  ✅ {path.name:<40}  {size_mb:.1f} MB")

    print()
    print("Loading best VGG16 model to verify...")
    vgg16_path = MODEL_DIR / "vgg16_phase2_best.h5"
    if vgg16_path.exists():
        loaded = keras.models.load_model(str(vgg16_path))
        dummy  = tf.random.normal((1, 224, 224, 3))
        out    = loaded.predict(dummy, verbose=0)
        print(f"  Output shape : {out.shape}")
        print(f"  Output value : {out[0][0]:.4f}  (probability of Dog)")
        print(f"  VGG16 loads and predicts correctly ✅")
    else:
        print("  vgg16_phase2_best.h5 not found — check Phase 2 training ran.")